In [1]:
import open3d as o3d

# List of filenames for your meshes
mesh_files = ['Meshes/PLY/mesh_1_2.ply', 'Meshes/PLY/mesh_1_3.ply']

# Load and visualize meshes
for file in mesh_files:
    try:
        mesh = o3d.io.read_triangle_mesh(file)  # Read mesh
        if not mesh.is_empty():  # Check if the mesh is valid
            print(f"Successfully loaded {file}")
            o3d.visualization.draw_geometries([mesh])  # Visualize the mesh
        else:
            print(f"Mesh {file} is empty.")
    except Exception as e:
        print(f"Error reading {file}: {e}")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Successfully loaded Meshes/PLY/mesh_1_2.ply
Successfully loaded Meshes/PLY/mesh_1_3.ply


In [2]:
import open3d as o3d

# List of filenames for your meshes
mesh_files = ['Meshes/PLY/mesh_1_2.ply', 'Meshes/PLY/mesh_1_3.ply']

# Load meshes as TriangleMesh and convert them to PointClouds
meshes = []
for file in mesh_files:
    try:
        mesh = o3d.io.read_triangle_mesh(file)  # Read mesh
        if not mesh.is_empty():  # Check if the mesh is valid
            print(f"Successfully loaded {file}")
            # Sample points uniformly from the mesh to convert it into a point cloud
            pcd = mesh.sample_points_uniformly(number_of_points=10000)  # Number of points to sample
            meshes.append(pcd)
            # Visualize the point cloud
            o3d.visualization.draw_geometries([pcd])
        else:
            print(f"Mesh {file} is empty.")
    except Exception as e:
        print(f"Error reading {file}: {e}")


Successfully loaded Meshes/PLY/mesh_1_2.ply
Successfully loaded Meshes/PLY/mesh_1_3.ply


In [3]:
# If meshes need to be aligned, use ICP (example of aligning second mesh to the first mesh)
reference_pcd = meshes[0]  # Take the first point cloud as reference
aligned_meshes = [reference_pcd]  # Start with the reference mesh

for i in range(1, len(meshes)):
    reg_icp = o3d.pipelines.registration.registration_icp(
        meshes[i], reference_pcd, max_correspondence_distance=0.02,
        criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
    )
    # Apply the ICP transformation
    meshes[i].transform(reg_icp.transformation)
    aligned_meshes.append(meshes[i])  # Add the aligned mesh to the list

# Visualize the aligned point clouds
o3d.visualization.draw_geometries(aligned_meshes)


In [41]:
import numpy as np
import open3d as o3d

# Assuming `aligned_meshes` is a list of PointClouds (resampled meshes)
# Convert PointClouds to numpy arrays and stack them together
points_list = [np.asarray(pcd.points) for pcd in aligned_meshes]

# Ensure all meshes have the same number of points (this is important for averaging)
num_points = points_list[0].shape[0]  # Number of points in each mesh (assumed same for all meshes)

# Check if all meshes have the same number of points
for points in points_list:
    if points.shape[0] != num_points:
        raise ValueError(f"Meshes have different numbers of points. Expected {num_points}, but got {points.shape[0]}.")

# Compute the mean across all meshes for each corresponding point
mean_points = np.mean(np.stack(points_list), axis=0)

# Check the shape of mean_points to ensure it's (N, 3)
print("Shape of mean_points:", mean_points.shape)

# Convert the mean points into an Open3D PointCloud
mean_pcd = o3d.geometry.PointCloud()
mean_pcd.points = o3d.utility.Vector3dVector(mean_points)

# Optionally, save or visualize the averaged point cloud
o3d.io.write_point_cloud("average_shape.ply", mean_pcd)
o3d.visualization.draw_geometries([mean_pcd])




Shape of mean_points: (10000, 3)


In [23]:
import open3d as o3d

# Step 1: Estimate normals for the point cloud
mean_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

# Step 2: Perform Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(mean_pcd, depth=9)

# Step 3: Apply Laplacian smoothing to the mesh
mesh = mesh.filter_smooth_laplacian(number_of_iterations=10)

# Visualize the point cloud with normals
o3d.visualization.draw_geometries([mean_pcd], point_show_normal=True)


# Step 4: Visualize the smoothed mesh
o3d.visualization.draw_geometries([mesh])

# Optionally, save the smoothed mesh
o3d.io.write_triangle_mesh("smoothed_average_shape.ply", mesh)




True

In [7]:
import open3d as o3d
import numpy as np
from pycpd import DeformableRegistration

# List of filenames for your meshes
mesh_files = ['Meshes/PLY/mesh_1_2.ply', 'Meshes/PLY/mesh_1_3.ply']

# Load meshes as TriangleMesh and convert them to PointClouds
meshes = []
for file in mesh_files:
    try:
        mesh = o3d.io.read_triangle_mesh(file)  # Read mesh
        if not mesh.is_empty():  # Check if the mesh is valid
            print(f"Successfully loaded {file}")
            # Sample points uniformly from the mesh to convert it into a point cloud
            pcd = mesh.sample_points_uniformly(number_of_points=10000)  # Number of points to sample
            meshes.append(pcd)
            # Visualize the point cloud
            o3d.visualization.draw_geometries([pcd])
        else:
            print(f"Mesh {file} is empty.")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Convert PointClouds to numpy arrays
points_list = [np.asarray(pcd.points) for pcd in meshes]

# Use the first point cloud as the reference
reference_points = points_list[0]

# Perform non-rigid registration using CPD
aligned_points_list = [reference_points]
for i in range(1, len(points_list)):
    moving_points = points_list[i]
    reg = DeformableRegistration(X=reference_points, Y=moving_points)
    TY, _ = reg.register()
    aligned_points_list.append(TY)

# Compute the mean across all aligned point clouds for each corresponding point
mean_points = np.mean(np.stack(aligned_points_list), axis=0)

# Check the shape of mean_points to ensure it's (N, 3)
print("Shape of mean_points:", mean_points.shape)

# Convert the mean points into an Open3D PointCloud
mean_pcd = o3d.geometry.PointCloud()
mean_pcd.points = o3d.utility.Vector3dVector(mean_points)

# Optionally, save or visualize the averaged point cloud
o3d.io.write_point_cloud("average_shape.ply", mean_pcd)
o3d.visualization.draw_geometries([mean_pcd])


Successfully loaded Meshes/PLY/mesh_1_2.ply
Successfully loaded Meshes/PLY/mesh_1_3.ply


KeyboardInterrupt: 

In [34]:
import open3d as o3d
import numpy as np
import geomstats.backend as gs
from geomstats.geometry.pre_shape import PreShapeSpace
from geomstats.learning.frechet_mean import FrechetMean
from scipy.spatial import cKDTree

# List of filenames for your meshes
mesh_files = ['Meshes/PLY/mesh_1_2.ply', 'Meshes/PLY/mesh_1_3.ply']

# Desired number of vertices for remeshing
desired_num_vertices = 100000

# Load and remesh meshes as TriangleMesh
meshes = []
for file in mesh_files:
    try:
        mesh = o3d.io.read_triangle_mesh(file)  # Read mesh
        if not mesh.is_empty():  # Check if the mesh is valid
            print(f"Successfully loaded {file}")
            # Simplify the mesh to the desired number of vertices
            mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=desired_num_vertices)
            meshes.append(mesh)
            # Visualize the mesh
            o3d.visualization.draw_geometries([mesh])
        else:
            print(f"Mesh {file} is empty.")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Convert meshes to numpy arrays of vertices
vertices_list = [np.asarray(mesh.vertices) for mesh in meshes]

# Ensure all point clouds have the same number of vertices by interpolating
num_vertices = min(vertices.shape[0] for vertices in vertices_list)
interpolated_vertices_list = []

for vertices in vertices_list:
    if vertices.shape[0] != num_vertices:
        tree = cKDTree(vertices)
        _, idx = tree.query(vertices[:num_vertices])
        interpolated_vertices = vertices[idx]
    else:
        interpolated_vertices = vertices
    interpolated_vertices_list.append(interpolated_vertices)

# Convert vertices to Geomstats format
vertices_array = gs.array(interpolated_vertices_list)

# Create a PreShapeSpace for shape analysis
preshape_space = PreShapeSpace(k_landmarks=num_vertices, ambient_dim=3)

# Compute the mean shape using FrechetMean with custom optimizer parameters
frechet_mean = FrechetMean(space=preshape_space)
frechet_mean.optimizer.max_iter = 5000  # Set maximum number of iterations
frechet_mean.optimizer.tolerance = 1e-5  # Set tolerance for convergence
mean_shape = frechet_mean.fit(vertices_array).estimate_

# Convert the mean shape back to an Open3D TriangleMesh
mean_mesh = o3d.geometry.TriangleMesh()
mean_mesh.vertices = o3d.utility.Vector3dVector(mean_shape)
mean_mesh.triangles = meshes[0].triangles  # Use the triangles from the first mesh

# Optionally, save or visualize the mean shape mesh
o3d.io.write_triangle_mesh("mean_shape.ply", mean_mesh)
o3d.visualization.draw_geometries([mean_mesh])

Successfully loaded Meshes/PLY/mesh_1_2.ply
Successfully loaded Meshes/PLY/mesh_1_3.ply


In [33]:
import open3d as o3d
import numpy as np
import geomstats.backend as gs
from geomstats.geometry.pre_shape import PreShapeSpace
from geomstats.learning.frechet_mean import FrechetMean
from scipy.spatial import cKDTree

# List of filenames for your meshes
mesh_files = ['Meshes/PLY/mesh_1_2.ply', 'Meshes/PLY/mesh_1_3.ply']

# Desired number of vertices for remeshing
desired_num_vertices = 10000

# Load and remesh meshes as TriangleMesh
meshes = []
for file in mesh_files:
    try:
        mesh = o3d.io.read_triangle_mesh(file)  # Read mesh
        if not mesh.is_empty():  # Check if the mesh is valid
            print(f"Successfully loaded {file}")
            # Simplify the mesh to the desired number of vertices
            mesh = mesh.simplify_quadric_decimation(target_number_of_triangles=desired_num_vertices)
            print(f"Mesh {file} has {len(mesh.vertices)} vertices after decimation.")
            # Ensure uniform sampling
            mesh = mesh.sample_points_uniformly(number_of_points=desired_num_vertices)
            meshes.append(mesh)
        else:
            print(f"Mesh {file} is empty.")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Convert meshes to numpy arrays of vertices
vertices_list = [np.asarray(mesh.points) for mesh in meshes]

# Ensure all point clouds have the same number of vertices by interpolation
num_vertices = min(vertices.shape[0] for vertices in vertices_list)
aligned_vertices_list = []

for vertices in vertices_list:
    if vertices.shape[0] != num_vertices:
        tree = cKDTree(vertices)
        _, idx = tree.query(vertices[:num_vertices])
        interpolated_vertices = vertices[idx]
    else:
        interpolated_vertices = vertices
    aligned_vertices_list.append(interpolated_vertices)

# Convert vertices to Geomstats format
vertices_array = gs.array(aligned_vertices_list)

# Create a PreShapeSpace for shape analysis
preshape_space = PreShapeSpace(k_landmarks=num_vertices, ambient_dim=3)

# Align shapes before computing mean
aligned_vertices_array = gs.array(
    [preshape_space.metric.procrustes_analysis(v, vertices_array[0]) for v in vertices_array]
)


# Compute the mean shape using FrechetMean with custom optimizer parameters
frechet_mean = FrechetMean(space=preshape_space)
frechet_mean.optimizer.max_iter = 10000  # Increase max iterations
frechet_mean.optimizer.tolerance = 1e-6  # Reduce tolerance for better precision
frechet_mean.optimizer.learning_rate = 0.01  # Adjust step size
mean_shape = frechet_mean.fit(aligned_vertices_array, init=vertices_array[0]).estimate_

# Convert the mean shape back to an Open3D PointCloud
mean_pcd = o3d.geometry.PointCloud()
mean_pcd.points = o3d.utility.Vector3dVector(mean_shape)

# Reconstruct mesh using Poisson reconstruction
mean_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(mean_pcd, depth=9)
mean_mesh.compute_vertex_normals()

# Optionally, save or visualize the mean shape mesh
o3d.io.write_triangle_mesh("mean_shape.ply", mean_mesh)
o3d.visualization.draw_geometries([mean_mesh])


Successfully loaded Meshes/PLY/mesh_1_2.ply
Mesh Meshes/PLY/mesh_1_2.ply has 4926 vertices after decimation.
Successfully loaded Meshes/PLY/mesh_1_3.ply
Mesh Meshes/PLY/mesh_1_3.ply has 4925 vertices after decimation.


AttributeError: 'PreShapeMetric' object has no attribute 'procrustes_analysis'